![问题经稠密向量和稀疏向量两路召回，再融合排序为统一结果列表](assets/multi-retrieval-fusion-ranking.png)

# LLM Reranker

RRF 按名次融合，weighted 按分数融合；模型重排直接读取问题和候选正文，判断相关性。

## 配置 LiteLLM 网关

- 模型名称：`qwen3-rerank`
- API Base: `https://llm-bxsnzgjeej076kci.cn-beijing.maas.aliyuncs.com/compatible-api/v1`
- API Key: `自行配置`
- 分组名：`rerank`

## Rerank 接口测试

In [6]:
import httpx

RERANK_API_KEY = "sk-zCjzfRyCk5KGoEzLvhK20Q"  # 填写 LiteLLM 网关的 API KEY
RERANK_ENDPOINT = "http://localhost:4000/rerank"  # LiteLLM使用Cohere的兼容接口
RERANK_MODEL = "rerank"

In [7]:
question = "你有没有 A200 的产品图？"

# 教学候选：同型号参数、其他型号图库、同型号视觉素材、同型号售后。
documents = [
    "A200 产品参数：额定功率 200W，支持三档调节，整机重量 1.5kg。",
    "A300 产品图库：包含正面、侧面和使用场景图，可供宣传物料使用。",
    "A200 视觉素材包：包含白底主图、细节特写和场景展示图，可下载用于商品详情页。",
    "A200 售后流程：提交订单号后，客服会协助处理维修和退换货。",
]

In [8]:
async with httpx.AsyncClient(timeout=60) as client:
    response = await client.post(
        RERANK_ENDPOINT,
        headers={"Authorization": f"Bearer {RERANK_API_KEY}"},
        json={
            "model": RERANK_MODEL,
            "query": question,
            "documents": documents,
            "top_n": 4,
        },
    )
    response.raise_for_status()
    payload = response.json()

payload

{'id': 'd723ce73-351f-9c8d-bd89-18a3a4375412',
 'results': [{'index': 2, 'relevance_score': 0.9557132901348567},
  {'index': 0, 'relevance_score': 0.8715499708093957},
  {'index': 1, 'relevance_score': 0.7381807563024175},
  {'index': 3, 'relevance_score': 0.6559366881101465}],
 'meta': {'billed_units': {'total_tokens': 206},
  'tokens': {'input_tokens': 206}}}

## 使用 Milvus 模型排序

**两路召回 → Milvus 调用 rerank → 返回 Top K**

运行后续代码前先配置好 `docker/milvus/user.yaml`

然后执行：`docker compose restart milvus`

In [9]:
from deal_pilot.rag.store.feishu_doc.get_store import get_store
from pymilvus import Function, FunctionType

store = await get_store()

In [10]:
model_reranker = Function(
    name="qwen_rerank",
    input_field_names=["text"],  # 用于评分的正文字段
    function_type=FunctionType.RERANK,
    params={
        "reranker": "model",
        "provider": "cohere",
        "model_name": "rerank",
        "queries": [question],
    },
)

results = await store.asimilarity_search_with_score(
    query=question,
    fetch_k=20,  # 每路最多召回 20 条候选
    k=4,  # 模型排序后最多返回 4 条
    reranker=model_reranker,
)

[{"doc": doc, "score": score} for doc, score in results]

[{'doc': Document(metadata={'block_id': 'doxcn33rjtSOYOZd70hFRmnasPe', 'obj_token': 'VS58dlCTxoyH8jxGBD9cQfMpn2c', 'revision_id': 4, 'block_type': 27, 'space_id': '7699429734026841315', 'node_token': 'VFuRw24RuidRo7krGo2cIqGwngD', 'chunk_index': 0, 'url': 'https://www.feishu.cn/wiki/VFuRw24RuidRo7krGo2cIqGwngD', 'kind': 'image', 'node_title': 'PROD-001_A200产品与配置说明', 'order': 7, 'space_title': '产品与配置资料', 'table_row_index': None, 'parent_block_id': 'doxcnSDTcNLWZ3JCFmuuzcNl3oh', 'chunk_count': 1, 'parent_node_token': None, 'id': 'doxcn33rjtSOYOZd70hFRmnasPe', 'references': [], 'payload': {'bucket_name': 'feishu-images', 'bucket_filename': 'doxcn33rjtSOYOZd70hFRmnasPe.png'}, 'title_path': ['PROD-001_A200产品与配置说明', 'A200 推理一体机产品与配置说明', '产品外观与内部照片']}, page_content='# PROD-001_A200产品与配置说明\n\n## A200 推理一体机产品与配置说明\n\n### 产品外观与内部照片\n\n这张图片展示了A200推理一体机（四卡版）移除顶盖后的内部结构及前面板外观，用于销售选型和部件位置沟通，以便核对设备配置。\n\n图中显示了一台银灰色的机架式服务器设备，机箱内部组件清晰可见。\n\n在机箱内部顶端（后部），并排安装有两个电源模块。左侧电源模块上的标签文字为“PSU 1”，下方有一行带有三角形感叹号警告图标的